# 04. Baseline: новый профиль -> существующий или новый `entity_id`

Это минимальный baseline-ноутбук.

```text
новый profile_id -> найти исторический entity_id или создать новый entity_id
```

Внутри всё равно используется pair-модель: она оценивает пару `новый профиль + исторический профиль`. Затем scores агрегируются до уровня `entity_id`.

In [ ]:
from pathlib import Path
from itertools import combinations
import hashlib
import pickle

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 180)

DATA_DIR = Path('../data/processed/er_profile_mart_multivalue')
OUTPUT_DIR = Path('../data/processed/er_baseline_pair_model')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
# Если брать все пары из каждого большого блока, train раздуется до миллионов шумных пар. Поэтому для
# обучения мы говорим: из одного блока брать максимум 800 пар (берём случайные)
MAX_PAIRS_PER_BLOCK = 800
# После того как мы собрали пары из блоков, они делятся на: positive pair: два профиля с одинаковым
# entity_id negative pair: два профиля с разным entity_id. Negative pairs намного больше. Если учить на
# всех negative pairs, модель и ноутбук будут тяжелее, а baseline не станет принципиально лучше. Поэтому мы
# ограничиваем: берём максимум 120 000 negative pairs для обучения Positive pairs при этом стараемся сохранить все, которые blocking смог поймать.
MAX_NEGATIVE_PAIRS = 120_000
# Сколько singleton-профилей брать в assignment-симуляцию как “реально новых клиентов”.
# То есть профилей, у которых в истории нет другого профиля с тем же entity_id. Мы ограничиваем их 2 000, чтобы assignment-проверка не раздувалась.
MAX_ASSIGNMENT_NEW_PROFILES = 2_000
# Размер чанка для скоринга полного candidate universe.Например, в test у нас 847 000 candidate pairs. Мы не считаем все признаки и score одним огромным куском, а обрабатываем пачками по 100 000 пар, чтобы не упереться в память.
FULL_PAIR_SCORE_CHUNK_SIZE = 100_000
# Минимальное число разных blocking-правил, через которые должна быть найдена пара, чтобы мы вообще скорили
# её моделью. Если пара найдена только одним правилом, например только через fallback, она считается слишком слабой и получает score 0 в full-universe оценке.
PAIR_QUALITY_GATE_MIN_RULES = 2
rng = np.random.default_rng(RANDOM_SEED)


## 1. Загружаем витрину и blocking

Берём только артефакты из предыдущего ноутбука. Ключевая проверка здесь — `blocking_model_index` обязан покрывать 100% профилей. Если coverage меньше 1.0, baseline дальше запускать нельзя: часть новых профилей вообще не сможет получить кандидатов.

`blocking_model_index coverage` — доля всех `profile_id`, которые попали хотя бы в один используемый blocking-блок. Это техническая метрика охвата профилей, а не качество поиска дублей.

In [ ]:
profile_core = pd.read_parquet(DATA_DIR / 'profile_core.parquet')
profile_value_summary_long = pd.read_parquet(DATA_DIR / 'profile_value_summary_long.parquet')
blocking_index = pd.read_parquet(DATA_DIR / 'blocking_index.parquet')
recommended_rules = pd.read_csv(DATA_DIR / 'recommended_blocking_rules.csv')
quality_check_report = pd.read_csv(DATA_DIR / 'quality_check_report.csv')
blocking_coverage_report = pd.read_csv(DATA_DIR / 'blocking_profile_coverage_report.csv')

failed_checks = quality_check_report[quality_check_report['status'].ne('ok')]
assert failed_checks.empty, failed_checks
assert blocking_coverage_report['profile_coverage'].min() == 1.0, blocking_coverage_report

selected_rules = recommended_rules[recommended_rules['recommended_for_next_step']][['block_family', 'block_rule']].drop_duplicates()
rule_index = pd.MultiIndex.from_frame(blocking_index[['block_family', 'block_rule']])
selected_index = pd.MultiIndex.from_frame(selected_rules)
blocking_model_index = blocking_index[rule_index.isin(selected_index)].copy()

block_size_lookup = (
    blocking_model_index
    .groupby(['block_family', 'block_rule', 'block_value'], sort=False)['profile_id']
    .nunique()
    .rename('block_size')
    .reset_index()
)
block_size_lookup['block_weight'] = 1 / np.log1p(block_size_lookup['block_size'])
blocking_model_index = blocking_model_index.merge(
    block_size_lookup,
    on=['block_family', 'block_rule', 'block_value'],
    how='left',
)

model_index_coverage = blocking_model_index['profile_id'].nunique() / profile_core['profile_id'].nunique()
assert model_index_coverage == 1.0, model_index_coverage

print('profiles:', profile_core['profile_id'].nunique())
print('selected blocking rules:', len(selected_rules))
print('blocking_model_index:', blocking_model_index.shape)
print('blocking_model_index coverage:', model_index_coverage)
display(selected_rules.sort_values(['block_family', 'block_rule']))


## 2. Candidate pairs для обучения

Candidate pairs — это пары профилей, найденные через общий blocking-блок. Мы не перебираем все пары профилей, потому что это квадратичный перебор.

Для обучения берём sampled negative pairs и обязательно добавляем все positive pairs, которые выбранный blocking умеет поймать. Это не label leakage: модель не получает `entity_id` как признак, но такая процедура делает pair-метрики оптимистичными относительно полного production-потока, потому что негативы сэмплируются, а positives сохраняются полностью.

`blocking_pair_recall_in_model_index` — доля настоящих positive pairs, которые вообще можно получить через выбранный blocking. Если настоящая пара не попала в кандидаты, pair-модель её уже не увидит.

In [ ]:
# делит данные на train / valid / test по entity_id, а не случайно по парам или профилям. Это нужно, чтобы
# одна и та же сущность не попала одновременно в обучение и проверку. Технически функция берёт стабильный hash от entity_id и по bucket-у относит сущность в один из split-ов, поэтому разбиение воспроизводимое при каждом запуске.
def stable_entity_split(entity_id):
    bucket = int(hashlib.md5(str(entity_id).encode('utf-8')).hexdigest()[:8], 16) % 100
    if bucket < 70:
        return 'train'
    if bucket < 85:
        return 'valid'
    return 'test'

# риводит пару профилей к единому порядку: меньший profile_id всегда слева, больший справа. Это нужно,
# чтобы пары (A, B) и (B, A) считались одной и той же парой, а не двумя разными наблюдениями.
def canonical_pair(left, right):
    left, right = str(left), str(right)
    return (left, right) if left < right else (right, left)

"""
Cтроит все настоящие positive pairs по entity_id.
Если у одной сущности несколько профилей, например:
entity_id = E1
profiles = A, B, C
то функция создаёт пары:
A-B
A-C
B-C
Эти пары получают label = 1, потому что оба профиля внутри пары относятся к одному и тому же entity_id.
"""
def make_positive_pairs(core):
    rows = []
    for entity_id, grp in core.groupby('entity_id', sort=False):
        profiles = sorted(map(str, grp['profile_id'].unique()))
        if len(profiles) >= 2:
            rows.extend((entity_id, left, right) for left, right in combinations(profiles, 2))
    out = pd.DataFrame(rows, columns=['entity_id', 'profile_id_l', 'profile_id_r'])
    out['pair_key'] = out['profile_id_l'] + '|' + out['profile_id_r']
    return out

"""
Берёт blocking-блоки и превращает их в пары профилей для обучения. Если внутри блока мало профилей, функция
 берёт все возможные пары. Если блок большой и пар слишком много, она случайно сэмплирует не больше MAX_PAIRS_PER_BLOCK, чтобы один крупный блок не раздул train и не забил его шумными negative pairs.
Пример:
10 профилей в блоке -> 45 пар, берём все
1000 профилей в блоке -> 499 500 пар, берём максимум 800
"""
def sample_pairs_from_blocks(index_df):
    rows = []
    group_cols = ['block_family', 'block_rule', 'block_value']
    for (family, rule, value), grp in index_df.groupby(group_cols, sort=False):
        profiles = np.array(sorted(map(str, grp['profile_id'].unique())))
        n = len(profiles)
        if n < 2:
            continue
        block_size = int(grp['block_size'].iloc[0])
        block_weight = float(grp['block_weight'].iloc[0])
        full_pairs = n * (n - 1) // 2
        if full_pairs <= MAX_PAIRS_PER_BLOCK:
            rows.extend((profiles[i], profiles[j], family, rule, value, block_size, block_weight) for i in range(n - 1) for j in range(i + 1, n))
        else:
            sampled = set()
            while len(sampled) < MAX_PAIRS_PER_BLOCK:
                draw = rng.choice(n, size=(MAX_PAIRS_PER_BLOCK - len(sampled)) * 2, replace=True)
                for k in range(0, len(draw) - 1, 2):
                    if draw[k] != draw[k + 1]:
                        sampled.add(canonical_pair(profiles[int(draw[k])], profiles[int(draw[k + 1])]))
                    if len(sampled) >= MAX_PAIRS_PER_BLOCK:
                        break
            rows.extend((left, right, family, rule, value, block_size, block_weight) for left, right in sampled)
    return pd.DataFrame(rows, columns=['profile_id_l', 'profile_id_r', 'block_family', 'block_rule', 'block_value', 'block_size', 'block_weight'])

"""
Отдельно добавляет в обучение все positive pairs, которые выбранный blocking_index вообще способен найти.
Это нужно потому, что sample_pairs_from_blocks в больших блоках берёт только случайные 800 пар и может случайно пропустить настоящую positive pair. Поэтому мы явно возвращаем такие пары в train, чтобы не потерять полезные positive-примеры при сэмплинге.
"""
def captured_positive_pairs(index_df, positive_pairs):
    rows = []
    group_cols = ['block_family', 'block_rule']
    base_cols = ['profile_id', 'block_family', 'block_rule', 'block_value', 'block_size', 'block_weight']
    base = index_df[base_cols].drop_duplicates()
    for (family, rule), rule_idx in base.groupby(group_cols, sort=False):
        left_idx = rule_idx[['profile_id', 'block_value', 'block_size', 'block_weight']].rename(columns={'profile_id': 'profile_id_l'})
        right_idx = rule_idx[['profile_id', 'block_value']].rename(columns={'profile_id': 'profile_id_r'})
        left = positive_pairs.merge(left_idx, on='profile_id_l')
        hit = left.merge(right_idx, on=['profile_id_r', 'block_value'])
        if not hit.empty:
            rows.append(
                hit[['profile_id_l', 'profile_id_r', 'block_value', 'block_size', 'block_weight']]
                .assign(block_family=family, block_rule=rule)
            )
    if not rows:
        return pd.DataFrame(columns=['profile_id_l', 'profile_id_r', 'block_family', 'block_rule', 'block_value', 'block_size', 'block_weight'])
    return pd.concat(rows, ignore_index=True)

"""
Cхлопывает “события попадания пары в блоки” до одной строки на пару профилей.
Одна и та же пара может попасть в несколько blocking-правил.

То есть на входе много строк вида:
A-B найдено через rule_1
A-B найдено через rule_2
A-B найдено через rule_3
а на выходе одна строка:
A-B, сколько правил сработало, какие семейства сработали, насколько сильные были блоки
"""
def events_to_evidence(events, profile_to_entity):
    events = events.copy()
    events['pair_key'] = events['profile_id_l'] + '|' + events['profile_id_r']
    events['label'] = events['profile_id_l'].map(profile_to_entity).eq(events['profile_id_r'].map(profile_to_entity)).astype('int8')
    out = (
        events.groupby(['profile_id_l', 'profile_id_r', 'pair_key', 'label'], sort=False)
        .agg(
            n_block_rules=('block_rule', 'nunique'),
            n_block_families=('block_family', 'nunique'),
            min_block_size=('block_size', 'min'),
            sum_block_weight=('block_weight', 'sum'),
            hit_coverage_fallback=('block_family', lambda s: int((s == 'coverage_fallback').any())),
        )
        .reset_index()
    )
    out['is_fallback_only'] = ((out['hit_coverage_fallback'].eq(1)) & (out['n_block_families'].eq(1))).astype('int8')
    out['has_non_fallback_signal'] = (out['n_block_families'].gt(out['hit_coverage_fallback'])).astype('int8')
    return out

def pair_quality_gate(evidence_df):
    return evidence_df['n_block_rules'].ge(PAIR_QUALITY_GATE_MIN_RULES)


profile_to_entity = profile_core.set_index('profile_id')['entity_id'].to_dict()
positive_pairs = make_positive_pairs(profile_core)

sampled = sample_pairs_from_blocks(blocking_model_index)
positive_hits = captured_positive_pairs(blocking_model_index, positive_pairs)
pair_events = pd.concat([sampled, positive_hits], ignore_index=True)
pair_events = pair_events.drop_duplicates(['profile_id_l', 'profile_id_r', 'block_family', 'block_rule', 'block_value'])

pair_evidence_all = events_to_evidence(pair_events, profile_to_entity)
blocking_pair_recall = pair_evidence_all['label'].sum() / len(positive_pairs)
pair_evidence_all = pair_evidence_all[pair_quality_gate(pair_evidence_all)].copy()

positive_part = pair_evidence_all[pair_evidence_all['label'].eq(1)]
negative_part = pair_evidence_all[pair_evidence_all['label'].eq(0)]
if len(negative_part) > MAX_NEGATIVE_PAIRS:
    negative_part = negative_part.sample(MAX_NEGATIVE_PAIRS, random_state=RANDOM_SEED)

pair_evidence = pd.concat([positive_part, negative_part], ignore_index=True)
quality_gate_positive_recall_in_training_candidates = pair_evidence['label'].sum() / len(positive_pairs)

# Количество пар дублей.Пример: entity_id = E1 profiles = A, B
# даёт 1 positive pair: A-B. А если: entity_id = E2 profiles = C, D, E
# то это уже 3 positive pairs: C-D C-E D-E
print('positive pairs:', len(positive_pairs))
# Размер обучающего датасета для pair-классификатора.
print('training candidate pairs:', pair_evidence.shape)
print('label distribution:')
print(pair_evidence['label'].value_counts())
# показывает, какую долю настоящих дублей вообще смог найти выбранный blocking. Это верхняя граница для
# pair-модели. Если настоящая пара не попала в blocking_model_index, модель её уже не увидит и не сможет правильно предсказать.
print('blocking pair recall in model index:', blocking_pair_recall)
# показывает, какая доля всех настоящих positive pairs осталась после pair_quality_gate.
# Например Если значение около 0.865, это значит:
# 86.5% настоящих дублей дошли до обучения после quality gate
# То есть quality gate убирает шумные пары, но вместе с ними теряет часть настоящих дублей.

print('quality-gated positive recall in training candidates:', quality_gate_positive_recall_in_training_candidates)


In [ ]:
pair_evidence.head(1)

## 3. Pair-признаки

Модель не смотрит на один профиль отдельно. Она смотрит на пару профилей и получает признаки, которые
отвечают на вопрос: насколько эти два профиля похожи? Смотрим Jaccard similarity — насколько сильно
множества пересекаются относительно всех уникальных значений: A ∩ B = {site_2}
A ∪ B = {site_1, site_2, site_3}
Jaccard = 1 / 3 = 0.333. Jaccard — это `|A ∩ B| / |A ∪ B|`, то есть доля общих значений среди всех значений
 двух профилей.

Pair-признаки описывают сходство двух профилей: пересечения значений, Jaccard similarity и минимальный
набор evidence-признаков из blocking. То есть не сходство значений напрямую, а информация о том, как именно
 пара была найдена: n_block_rules = 3 значит пару подтвердили 3 разных blocking-правила. min_block_size = 2
  значит хотя бы одно совпадение было в очень маленьком блоке, а это сильный сигнал. is_fallback_only = 1
  значит пара нашлась только через технический fallback, такой сигнал слабее. Итого получаем набор чисел
  про пару: совпали ли значения? насколько сильно совпали? по
скольким blocking-правилам пара нашлась? насколько качественные были эти blocking-сигнал?
Если в production мы сначала строим тот же blocking index, потом по нему получаем candidate pairs, и только
 потом скорим пары моделью. Эти признаки будут доступны на inference.
Но они делают pair-метрики выше, потому что модель получает не только сходство значений, но и
мета-информацию: "насколько уверенно blocking нашел эту пару".
- `n_block_rules` — сколько разных blocking-правил подтвердило пару;
- `n_block_families` — сколько разных семейств blocking подтвердило пару;
- `min_block_size` — размер самого маленького блока, через который пара была найдена;
- `sum_block_weight`, где `block_weight = 1 / log1p(block_size)` — суммарная сила blocking-сигнала;
- `hit_coverage_fallback` — был ли среди сработавших правил технический fallback;
- `is_fallback_only` — пара найдена только через fallback;
- `has_non_fallback_signal` — у пары есть хотя бы один не-fallback сигнал.

Зачем это нужно: совпадение в блоке размера 2 намного сильнее, чем совпадение в блоке размера 900. При этом baseline не должен разрастаться десятками почти дублирующих blocking-флагов, поэтому оставляем только признаки, которые нужны для качества сигнала и quality gate.

Quality gate: дальше скорим только пары, которые найдены минимум двумя blocking-правилами (`n_block_rules >= 2`). Пары, не прошедшие gate, в full-universe оценке остаются в знаменателе, но получают score `0`, то есть не могут стать predicted duplicate.

In [ ]:
PAIR_FEATURES = [
    ('identity', 'email'), ('identity', 'phone'), ('identity', 'first_name'), ('identity', 'last_name'), ('identity', 'birthday'), ('identity', 'sex'),
    ('np', 'geoname_id'), ('np', 'subdivision_1_iso_code'), ('np', 'device'), ('np', 'browser'), ('np', 'osfamily'),
    ('rt', 'geoid'), ('rt', 'geoname'), ('rt', 'country'),
    ('fs', 'source_site_365'), ('fs', 'source_site_30'), ('fs', 'visited_30'), ('fs', 'visited_365'), ('fs', 'has_account'), ('fs', 'has_click_365'), ('fs', 'has_accept_365'),
]

value_maps = {}
for source, feature in PAIR_FEATURES:
    key = f'{source}__{feature}'
    sub = profile_value_summary_long.loc[
        (profile_value_summary_long['source'] == source) & (profile_value_summary_long['feature'] == feature),
        ['profile_id', 'value_norm'],
    ].drop_duplicates()
    value_maps[key] = sub.groupby('profile_id', observed=True)['value_norm'].agg(lambda values: set(map(str, values))).to_dict()

geo_keys = {'np__geoname_id', 'np__subdivision_1_iso_code', 'rt__geoid', 'rt__geoname'}


def overlap(left_values, right_values):
    if not left_values or not right_values:
        return 0, 0.0, 0
    intersection = len(left_values & right_values)
    union = len(left_values | right_values)
    return intersection, intersection / union if union else 0.0, 1


def build_pair_features(evidence_df):
    rows = []
    for row in evidence_df.itertuples(index=False):
        features = {
            'profile_id_l': row.profile_id_l,
            'profile_id_r': row.profile_id_r,
            'pair_key': row.pair_key,
            'label': row.label,
            'n_block_rules': row.n_block_rules,
            'n_block_families': row.n_block_families,
            'min_block_size': row.min_block_size,
            'sum_block_weight': row.sum_block_weight,
            'hit_coverage_fallback': row.hit_coverage_fallback,
            'is_fallback_only': row.is_fallback_only,
            'has_non_fallback_signal': row.has_non_fallback_signal,
        }

        fs_intersection = fs_union = geo_intersection = geo_union = 0
        for key, profile_values in value_maps.items():
            left = profile_values.get(row.profile_id_l, set())
            right = profile_values.get(row.profile_id_r, set())
            intersection, jaccard, both_present = overlap(left, right)
            features[f'{key}__intersect_count'] = intersection
            features[f'{key}__jaccard'] = jaccard
            features[f'{key}__both_present'] = both_present
            if key.startswith('identity__'):
                features[f'{key}__conflict'] = int(both_present and intersection == 0)
            if key.startswith('fs__'):
                fs_intersection += intersection
                fs_union += len(left | right)
            if key in geo_keys:
                geo_intersection += intersection
                geo_union += len(left | right)
        features['fs_total_jaccard'] = fs_intersection / fs_union if fs_union else 0.0
        features['geo_total_jaccard'] = geo_intersection / geo_union if geo_union else 0.0
        rows.append(features)
    return pd.DataFrame(rows)

pair_features = build_pair_features(pair_evidence)
print('pair_features:', pair_features.shape)


## 4. Обучаем pair-модель

Используем логистическую регрессию как простой baseline. Разбиение делаем по `entity_id`, чтобы одна и та же сущность не попадала одновременно в train и test.

На sampled candidate pairs модель только обучается. Метрики качества на этих парах намеренно не считаем: после negative sampling это искусственный датасет, и его `PR-AUC` / `ROC-AUC` легко переоценить.

В этом блоке сохраняем только техническую сводку обучения: сколько признаков и пар попало в train/valid/test, сколько positive pairs осталось после blocking и quality gate.


In [ ]:

profile_split = profile_core.assign(split=profile_core['entity_id'].map(stable_entity_split)).set_index('profile_id')['split'].to_dict()
model_table = pair_features.copy()
model_table['split_l'] = model_table['profile_id_l'].map(profile_split)
model_table['split_r'] = model_table['profile_id_r'].map(profile_split)
model_table['pair_split'] = np.where(model_table['split_l'].eq(model_table['split_r']), model_table['split_l'], 'cross_split')
model_table = model_table[model_table['pair_split'].isin(['train', 'valid', 'test'])].copy()

feature_cols = [c for c in model_table.columns if c not in ['profile_id_l', 'profile_id_r', 'pair_key', 'label', 'split_l', 'split_r', 'pair_split']]
train_df = model_table[model_table['pair_split'].eq('train')]
valid_df = model_table[model_table['pair_split'].eq('valid')]
test_df = model_table[model_table['pair_split'].eq('test')]

model = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced')),
])
model.fit(train_df[feature_cols], train_df['label'])

model_training_summary = pd.DataFrame([
    {'metric': 'n_model_features', 'value': len(feature_cols)},
    {'metric': 'train_pairs_after_sampling_and_gate', 'value': len(train_df)},
    {'metric': 'valid_pairs_after_sampling_and_gate', 'value': len(valid_df)},
    {'metric': 'test_pairs_after_sampling_and_gate', 'value': len(test_df)},
    {'metric': 'train_positive_pairs', 'value': int(train_df['label'].sum())},
    {'metric': 'valid_positive_pairs', 'value': int(valid_df['label'].sum())},
    {'metric': 'test_positive_pairs', 'value': int(test_df['label'].sum())},
    {'metric': 'blocking_model_profile_coverage', 'value': model_index_coverage},
    {'metric': 'blocking_pair_recall_in_model_index', 'value': blocking_pair_recall},
    {'metric': 'quality_gate_min_blocking_rules', 'value': PAIR_QUALITY_GATE_MIN_RULES},
    {'metric': 'quality_gated_positive_recall_in_training_candidates', 'value': quality_gate_positive_recall_in_training_candidates},
])

display(model_training_summary)


## 5. Проверка утечек

Здесь проверяем только явные leakage-риски:

- `entity_id`, `entity_size`, `entity_kind`, `profile_id`, `pair_key`, `label` не должны попасть в `feature_cols`;
- train/valid/test должны быть disjoint по `entity_id`;
- одинаковая `pair_key` не должна одновременно попасть в train и test.

Аналитику качества на sampled pairs и абляции по sampled test убрали: это дорогая и слишком искусственная проверка для baseline. Качество ниже оцениваем на полном test candidate universe и в assignment-сценарии.


In [ ]:

leakage_rows = []

split_entities = (
    profile_core[['entity_id']]
    .drop_duplicates()
    .assign(split=lambda df: df['entity_id'].map(stable_entity_split))
)
split_entity_sets = {split: set(split_entities.loc[split_entities['split'].eq(split), 'entity_id']) for split in ['train', 'valid', 'test']}
entity_overlap_count = sum(
    len(split_entity_sets[left] & split_entity_sets[right])
    for left, right in [('train', 'valid'), ('train', 'test'), ('valid', 'test')]
)

forbidden_exact_cols = {'entity_id', 'entity_size', 'entity_kind', 'label', 'profile_id_l', 'profile_id_r', 'pair_key', 'split_l', 'split_r', 'pair_split'}
forbidden_feature_cols = [col for col in feature_cols if col in forbidden_exact_cols or col.startswith('entity_')]
train_test_pair_overlap = len(set(train_df['pair_key']) & set(test_df['pair_key']))
valid_test_pair_overlap = len(set(valid_df['pair_key']) & set(test_df['pair_key']))

leakage_rows.extend([
    {
        'check_name': 'entity_disjoint_split',
        'status': 'ok' if entity_overlap_count == 0 else 'fail',
        'metric': entity_overlap_count,
        'expected': '0 overlapping entity_id between train/valid/test',
    },
    {
        'check_name': 'no_label_or_id_columns_in_features',
        'status': 'ok' if not forbidden_feature_cols else 'fail',
        'metric': ', '.join(forbidden_feature_cols) if forbidden_feature_cols else 'none',
        'expected': 'no entity_id/profile_id/pair_key/label columns in X',
    },
    {
        'check_name': 'no_train_test_pair_overlap',
        'status': 'ok' if train_test_pair_overlap == 0 else 'fail',
        'metric': train_test_pair_overlap,
        'expected': '0 shared pair_key between train and test',
    },
    {
        'check_name': 'no_valid_test_pair_overlap',
        'status': 'ok' if valid_test_pair_overlap == 0 else 'fail',
        'metric': valid_test_pair_overlap,
        'expected': '0 shared pair_key between valid and test',
    },
])

leakage_check_report = pd.DataFrame(leakage_rows)
display(leakage_check_report)
assert leakage_check_report['status'].eq('ok').all(), leakage_check_report


## 6. Честная pair-метрика на полном candidate universe

Здесь считаем pair-качество без negative sampling. Берём все пары, которые выбранный blocking создаёт внутри `test` split, скорим их моделью и считаем метрики на полном test candidate universe.

Почему только `test`, а не весь датасет: модель обучалась на `train`, а объективную проверку делаем на отложенном `test`.

Ключевые поля:

- `test_candidate_pairs_full_universe` — сколько уникальных candidate pairs реально создал blocking на test;
- `test_candidate_positive_pairs` — сколько среди них настоящих positive pairs;
- `test_total_true_pairs` — сколько всего true pairs существует в test, независимо от blocking;
- `test_blocking_pair_recall` — доля всех test true pairs, попавших в candidate universe;
- `test_quality_gate_recall_vs_all_true_pairs` — доля всех true pairs, которые прошли blocking и quality gate;
- `test_full_candidate_pr_auc` / `test_full_candidate_roc_auc` — ranking-качество на полном candidate universe, уже без negative sampling.


In [ ]:

def all_pairs_from_blocks(index_df):
    rows = []
    for (family, rule, value), grp in index_df.groupby(['block_family', 'block_rule', 'block_value'], sort=False):
        profiles = sorted(map(str, grp['profile_id'].unique()))
        if len(profiles) >= 2:
            block_size = int(grp['block_size'].iloc[0])
            block_weight = float(grp['block_weight'].iloc[0])
            rows.extend((left, right, family, rule, value, block_size, block_weight) for left, right in combinations(profiles, 2))
    return pd.DataFrame(rows, columns=['profile_id_l', 'profile_id_r', 'block_family', 'block_rule', 'block_value', 'block_size', 'block_weight'])


def score_pair_evidence_in_chunks(evidence_df):
    scores = []
    labels = []
    for start in range(0, len(evidence_df), FULL_PAIR_SCORE_CHUNK_SIZE):
        chunk = evidence_df.iloc[start:start + FULL_PAIR_SCORE_CHUNK_SIZE]
        chunk_features = build_pair_features(chunk)
        for col in feature_cols:
            if col not in chunk_features.columns:
                chunk_features[col] = 0
        scores.append(model.predict_proba(chunk_features[feature_cols])[:, 1])
        labels.append(chunk_features['label'].to_numpy())
    return np.concatenate(labels), np.concatenate(scores)


test_profile_ids = {profile_id for profile_id, split in profile_split.items() if split == 'test'}
test_blocking_index = blocking_model_index[blocking_model_index['profile_id'].isin(test_profile_ids)]

full_test_pair_events = all_pairs_from_blocks(test_blocking_index)
full_test_pair_evidence = events_to_evidence(full_test_pair_events, profile_to_entity)
full_test_quality_gate = pair_quality_gate(full_test_pair_evidence)
full_test_scores = np.zeros(len(full_test_pair_evidence), dtype='float64')
full_test_labels = full_test_pair_evidence['label'].to_numpy()
if full_test_quality_gate.any():
    _, gated_scores = score_pair_evidence_in_chunks(full_test_pair_evidence.loc[full_test_quality_gate])
    full_test_scores[full_test_quality_gate.to_numpy()] = gated_scores

all_test_positive_pairs = positive_pairs[
    positive_pairs['profile_id_l'].map(profile_split).eq('test') &
    positive_pairs['profile_id_r'].map(profile_split).eq('test')
]
total_test_true_pairs = len(all_test_positive_pairs)
full_candidate_true_pairs = int(full_test_labels.sum())
quality_gated_true_pairs = int(full_test_pair_evidence.loc[full_test_quality_gate, 'label'].sum())

full_candidate_pair_metrics = pd.DataFrame([
    {'metric': 'test_candidate_pairs_full_universe', 'value': len(full_test_pair_evidence)},
    {'metric': 'test_quality_gated_candidate_pairs', 'value': int(full_test_quality_gate.sum())},
    {'metric': 'test_candidate_positive_pairs', 'value': full_candidate_true_pairs},
    {'metric': 'test_quality_gated_positive_pairs', 'value': quality_gated_true_pairs},
    {'metric': 'test_total_true_pairs', 'value': total_test_true_pairs},
    {'metric': 'test_candidate_positive_rate', 'value': full_candidate_true_pairs / len(full_test_pair_evidence)},
    {'metric': 'test_blocking_pair_recall', 'value': full_candidate_true_pairs / total_test_true_pairs},
    {'metric': 'test_quality_gate_recall_vs_all_true_pairs', 'value': quality_gated_true_pairs / total_test_true_pairs},
    {'metric': 'test_quality_gate_recall_vs_candidate_true_pairs', 'value': quality_gated_true_pairs / full_candidate_true_pairs if full_candidate_true_pairs else 0.0},
    {'metric': 'test_full_candidate_pr_auc', 'value': average_precision_score(full_test_labels, full_test_scores)},
    {'metric': 'test_full_candidate_roc_auc', 'value': roc_auc_score(full_test_labels, full_test_scores)},
])

display(full_candidate_pair_metrics)


## 7. Проверяем assignment-сценарий

Симулируем production: часть профилей считаем новыми, остальные профили из того же split — историей. Для нового профиля ищем исторических кандидатов через blocking, скорим пары, агрегируем scores до `entity_id` и принимаем решение по threshold.

Threshold для assignment выбираем на `valid`, потому что assignment должен одновременно уметь присоединять профиль к существующему `entity_id` и отказывать, создавая новый `entity_id`. Pair-threshold на sampled pairs больше не используем.

Главные метрики assignment-уровня:

- `known_entity_true_candidate_coverage` — доля случаев, где правильный существующий `entity_id` вообще был среди кандидатов;
- `known_entity_assignment_recall` — доля профилей, которые правильно присоединились к существующему `entity_id`;
- `new_entity_rejection_rate` — доля реально новых профилей, для которых правильно создан новый `entity_id`;
- `overall_assignment_accuracy` — общий процент правильных assignment-решений;
- `balanced_assignment_accuracy` — среднее между `known_entity_assignment_recall` и `new_entity_rejection_rate`. Эта метрика полезна, когда новых entity больше, чем профилей, которые надо присоединить к истории.


In [ ]:
def build_assignment_set(split_name):
    split_profiles = profile_core.assign(split=profile_core['entity_id'].map(stable_entity_split)).query('split == @split_name').copy()
    entity_sizes = split_profiles.groupby('entity_id')['profile_id'].nunique()
    existing_entity_ids = entity_sizes[entity_sizes.ge(2)].index
    new_entity_ids = entity_sizes[entity_sizes.eq(1)].index

    known_new = (
        split_profiles[split_profiles['entity_id'].isin(existing_entity_ids)]
        .sort_values(['entity_id', 'profile_id'])
        .groupby('entity_id', sort=False)
        .head(1)[['profile_id', 'entity_id']]
        .rename(columns={'profile_id': 'new_profile_id', 'entity_id': 'true_entity_id'})
        .assign(expected_decision='existing_entity')
    )
    brand_new = (
        split_profiles[split_profiles['entity_id'].isin(new_entity_ids)]
        .sort_values('profile_id')
        .head(MAX_ASSIGNMENT_NEW_PROFILES)[['profile_id', 'entity_id']]
        .rename(columns={'profile_id': 'new_profile_id', 'entity_id': 'true_entity_id'})
        .assign(expected_decision='new_entity')
    )
    new_profiles = pd.concat([known_new, brand_new], ignore_index=True)
    history_profiles = split_profiles[~split_profiles['profile_id'].isin(set(new_profiles['new_profile_id']))].copy()
    return new_profiles, history_profiles


def score_assignment_split(split_name):
    new_profiles, history_profiles = build_assignment_set(split_name)
    new_idx = blocking_model_index[blocking_model_index['profile_id'].isin(set(new_profiles['new_profile_id']))]
    hist_idx = blocking_model_index[blocking_model_index['profile_id'].isin(set(history_profiles['profile_id']))]
    lookup_events = new_idx.rename(columns={'profile_id': 'profile_id_l'}).merge(
        hist_idx.rename(columns={'profile_id': 'profile_id_r'}),
        on=['block_family', 'block_rule', 'block_value'],
        how='inner',
    )
    lookup_events = lookup_events[lookup_events['profile_id_l'].ne(lookup_events['profile_id_r'])]
    lookup_events = lookup_events.merge(
        block_size_lookup,
        on=['block_family', 'block_rule', 'block_value'],
        how='left',
    )
    lookup_evidence = events_to_evidence(lookup_events[['profile_id_l', 'profile_id_r', 'block_family', 'block_rule', 'block_value', 'block_size', 'block_weight']], profile_to_entity)
    lookup_evidence = lookup_evidence[pair_quality_gate(lookup_evidence)].copy()
    if lookup_evidence.empty:
        result = new_profiles.copy()
        result['predicted_entity_id'] = pd.NA
        result['entity_max_score'] = 0.0
        result['candidate_profile_count'] = 0
        result['has_true_pair'] = 0
        result['has_true_candidate'] = 0
        result['split'] = split_name
        return result
    lookup_features = build_pair_features(lookup_evidence)
    for col in feature_cols:
        if col not in lookup_features.columns:
            lookup_features[col] = 0

    lookup_scores = lookup_features[['profile_id_l', 'profile_id_r', 'label']].rename(columns={'profile_id_l': 'new_profile_id', 'profile_id_r': 'history_profile_id'})
    lookup_scores['history_entity_id'] = lookup_scores['history_profile_id'].map(profile_to_entity)
    lookup_scores['score'] = model.predict_proba(lookup_features[feature_cols])[:, 1]

    entity_scores = (
        lookup_scores.groupby(['new_profile_id', 'history_entity_id'], sort=False)
        .agg(entity_max_score=('score', 'max'), candidate_profile_count=('history_profile_id', 'nunique'), has_true_pair=('label', 'max'))
        .reset_index()
    )
    best_entity = (
        entity_scores.sort_values(['new_profile_id', 'entity_max_score', 'candidate_profile_count'], ascending=[True, False, False])
        .drop_duplicates('new_profile_id')
        .rename(columns={'history_entity_id': 'predicted_entity_id'})
    )
    true_candidate = lookup_scores.groupby('new_profile_id')['label'].max().rename('has_true_candidate').reset_index()

    result = new_profiles.merge(best_entity, on='new_profile_id', how='left').merge(true_candidate, on='new_profile_id', how='left')
    result['entity_max_score'] = result['entity_max_score'].fillna(0.0)
    result['candidate_profile_count'] = result['candidate_profile_count'].fillna(0).astype(int)
    result['has_true_candidate'] = result['has_true_candidate'].fillna(0).astype(int)
    result['split'] = split_name
    return result


def evaluate_assignment(result, threshold):
    predicted_decision = np.where(result['entity_max_score'].ge(threshold), 'existing_entity', 'new_entity')
    is_correct = np.where(
        result['expected_decision'].eq('existing_entity'),
        (predicted_decision == 'existing_entity') & result['predicted_entity_id'].eq(result['true_entity_id']),
        predicted_decision == 'new_entity',
    )
    known_mask = result['expected_decision'].eq('existing_entity')
    new_mask = result['expected_decision'].eq('new_entity')
    known_recall = float(is_correct[known_mask].mean())
    new_rejection = float(is_correct[new_mask].mean())
    return {
        'threshold': float(threshold),
        'new_profiles_total': len(result),
        'known_existing_profiles': int(known_mask.sum()),
        'brand_new_profiles': int(new_mask.sum()),
        'known_entity_true_candidate_coverage': float(result.loc[known_mask, 'has_true_candidate'].mean()),
        'overall_assignment_accuracy': float(is_correct.mean()),
        'known_entity_assignment_recall': known_recall,
        'new_entity_rejection_rate': new_rejection,
        'balanced_assignment_accuracy': (known_recall + new_rejection) / 2,
    }

valid_assignment_base = score_assignment_split('valid')
test_assignment_base = score_assignment_split('test')

fixed_grid = np.array([0.2, 0.5, 0.8, 0.9, 0.95, 0.99, 0.999, 0.9999, 0.99999])
score_grid = np.quantile(valid_assignment_base['entity_max_score'], np.linspace(0.05, 0.99, 30))
threshold_grid = np.unique(np.clip(np.concatenate([fixed_grid, score_grid]), 0, 1))

valid_assignment_threshold_report = pd.DataFrame([evaluate_assignment(valid_assignment_base, t) for t in threshold_grid])
assignment_threshold = float(valid_assignment_threshold_report.sort_values('overall_assignment_accuracy', ascending=False).iloc[0]['threshold'])

valid_assignment_metrics = pd.DataFrame([{**evaluate_assignment(valid_assignment_base, assignment_threshold), 'split': 'valid'}])
test_assignment_metrics = pd.DataFrame([{**evaluate_assignment(test_assignment_base, assignment_threshold), 'split': 'test'}])
assignment_metrics = pd.concat([valid_assignment_metrics, test_assignment_metrics], ignore_index=True)

assignment_result = test_assignment_base.copy()
assignment_result['predicted_decision'] = np.where(assignment_result['entity_max_score'].ge(assignment_threshold), 'existing_entity', 'new_entity')
assignment_result['is_correct'] = np.where(
    assignment_result['expected_decision'].eq('existing_entity'),
    assignment_result['predicted_decision'].eq('existing_entity') & assignment_result['predicted_entity_id'].eq(assignment_result['true_entity_id']),
    assignment_result['predicted_decision'].eq('new_entity'),
)

display(assignment_metrics)


## 8. Сохраняем минимальные артефакты

Сохраняем только то, что нужно для baseline: техническую сводку обучения, честные full-candidate pair-метрики, leakage-check, assignment-результат и pickle с моделью, списком признаков и assignment-threshold.


In [ ]:

model_training_summary.to_csv(OUTPUT_DIR / 'model_training_summary.csv', index=False)
full_candidate_pair_metrics.to_csv(OUTPUT_DIR / 'full_candidate_pair_metrics.csv', index=False)
leakage_check_report.to_csv(OUTPUT_DIR / 'leakage_check_report.csv', index=False)
assignment_metrics.to_csv(OUTPUT_DIR / 'assignment_quality_report.csv', index=False)
assignment_result.to_parquet(OUTPUT_DIR / 'assignment_result.parquet', index=False)

with open(OUTPUT_DIR / 'baseline_assignment_model.pkl', 'wb') as f:
    pickle.dump({'model': model, 'feature_cols': feature_cols, 'assignment_threshold': assignment_threshold}, f)

current_outputs = [
    'model_training_summary.csv',
    'full_candidate_pair_metrics.csv',
    'leakage_check_report.csv',
    'assignment_quality_report.csv',
    'assignment_result.parquet',
    'baseline_assignment_model.pkl',
]
print('Current baseline outputs:')
for name in current_outputs:
    print('-', name)
